# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
%load_ext dotenv
%dotenv
import pandas as pd
import yfinance as yf
import os
import sys



In [2]:
import dask.dataframe as dd

sys.path.append(os.getenv('SRC_DIR'))

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

PRICE_DATA_PATH= "../../05_src/data/prices/"  

# Find all Parquet files recursively
parquet_files = glob(os.path.join(PRICE_DATA_PATH, "**", "*.parquet"), recursive=True)

# Read the Parquet files into a Dask DataFrame
PRICE_DATA = dd.read_parquet(parquet_files)

# Display the first few rows 
PRICE_DATA.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year
Ticker,,,,,,,,
A,2000-01-03 00:00:00+00:00,43.382851,51.502148,56.464592,48.193848,56.330471,4674353.0,2000
A,2000-01-04 00:00:00+00:00,40.068871,47.567955,49.266811,46.316166,48.730328,4765083.0,2000
A,2000-01-05 00:00:00+00:00,37.583397,44.617310,47.567955,43.141991,47.389126,5758642.0,2000
A,2000-01-06 00:00:00+00:00,36.152374,42.918453,44.349072,41.577251,44.080830,2534434.0,2000
A,2000-01-07 00:00:00+00:00,39.165073,46.494991,47.165592,42.203148,42.247852,2819626.0,2000


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
dd_shift = PRICE_DATA.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1=x["Close"].shift(1),
        Adj_Close_lag_1=x["Adj Close"].shift(1)
    )
)

# Compute returns and hi_lo_range in a single assign statement
dd_feat = dd_shift.assign(
    returns=lambda x: x["Close"] / x["Close_lag_1"] - 1,
    hi_lo_range=lambda x: x["High"] - x["Low"]
)

# Display the first few rows
dd_feat.head()


/var/folders/w_/10lptk3s1kjf61j34wyy_vtr0000gn/T/ipykernel_69072/2634134214.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = PRICE_DATA.groupby("Ticker", group_keys=False).apply(
/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/core.py:8153: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,,


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [19]:
# Convert Dask DataFrame to Pandas
pd_feat = dd_feat.compute()

In [25]:

# Add 10-day moving average for 'returns' per ticker
pd_feat["returns_ma_10"] = pd_feat.groupby("Ticker", group_keys=False)["returns"].transform(lambda x: x.rolling(10).mean())


# Display first few rows
pd_feat.head(20)

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
Ticker,,,,,,,,,,,,,
DOV,2003-01-02 00:00:00+00:00,13.402370,20.418425,20.773701,19.567097,19.627426,766183.0,2003,NaN,NaN,NaN,1.206604,NaN
DOV,2003-01-03 00:00:00+00:00,13.362778,20.358093,20.472050,20.170399,20.398314,624612.0,2003,20.418425,13.402370,-0.002955,0.301651,NaN
DOV,2003-01-06 00:00:00+00:00,13.732368,20.921175,21.068649,20.344687,20.344687,958028.0,2003,20.358093,13.362778,0.027659,0.723963,NaN
DOV,2003-01-07 00:00:00+00:00,13.587175,20.699965,20.947989,20.559195,20.921175,776626.0,2003,20.921175,13.732368,-0.010574,0.388794,NaN
DOV,2003-01-08 00:00:00+00:00,13.265974,20.210619,20.699965,20.177103,20.699965,659073.0,2003,20.699965,13.587175,-0.023640,0.522861,NaN
DOV,2003-01-09 00:00:00+00:00,13.644371,20.787107,20.813923,20.224026,20.224026,956237.0,2003,20.210619,13.265974,0.028524,0.589897,NaN
DOV,2003-01-10 00:00:00+00:00,13.481580,20.539083,20.854141,20.291059,20.378202,1219389.0,2003,20.787107,13.644371,-0.011932,0.563082,NaN
DOV,2003-01-13 00:00:00+00:00,13.393579,20.405018,20.787107,20.270950,20.612822,694428.0,2003,20.539083,13.481580,-0.006527,0.516157,NaN
DOV,2003-01-14 00:00:00+00:00,13.485974,20.545788,20.612822,20.244137,20.465347,615065.0,2003,20.405018,13.393579,0.006899,0.368685,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, converting to Pandas was not necessary because Dask can calculate the moving average directly. Doing it in Dask is better because it handles large datasets efficiently without using too much memory

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.